<a href="https://colab.research.google.com/github/MahathiKadiyala/24B11DS084_DriveReady_OPC_2026/blob/main/Day-3/Agentic_AI_DAY_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-groq

In [3]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("agentic_ai_GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq connected successfully!")

Groq connected successfully!


In [4]:
reply = llm.invoke("Do you make eggless cakes?")

print(reply.content)

Absolutely! Egg‑free cakes are a staple in many bakeries and can be made at home just as easily. Whether you’re looking for a quick recipe, a store‑bought option, or a custom order, there are plenty of ways to enjoy a delicious, eggless cake.

### Quick Eggless Cake Recipe (for 8‑10 servings)

| Ingredient | Amount |
|------------|--------|
| All‑purpose flour | 2 cups (≈240 g) |
| Granulated sugar | 1 cup (≈200 g) |
| Baking powder | 2 tsp |
| Baking soda | ½ tsp |
| Salt | ¼ tsp |
| Unsweetened applesauce | ½ cup (≈120 ml) |
| Milk (dairy or plant‑based) | ¾ cup (≈180 ml) |
| Vegetable oil | ½ cup (≈120 ml) |
| Vanilla extract | 1 tsp |
| Optional: cocoa powder (for chocolate) | ¼ cup (≈25 g) |

#### Instructions

1. **Preheat** the oven to 350 °F (175 °C). Grease and flour an 8‑inch round or square pan.
2. **Combine dry ingredients**: In a large bowl, whisk flour, sugar, baking powder, baking soda, salt, and cocoa (if using).
3. **Add wet ingredients**: Stir in applesauce, milk, oil

In [5]:
agent_parts = {
    "Brain": "LLM that understands and decides",
    "Instructions": "System prompt and business rules",
    "Tools": "Functions the agent can call",
    "Memory": "Conversation history"
}

for part, description in agent_parts.items():
    print(part, ":", description)

Brain : LLM that understands and decides
Instructions : System prompt and business rules
Tools : Functions the agent can call
Memory : Conversation history


In [6]:
steps = [
    "Think: What information do I need?",
    "Act: Call get_price('plum cake')",
    "Observe: Function returns Rs 500",
    "Think: Calculate the total for 3 cakes",
    "Answer: 3 plum cakes cost Rs 1500"
]

for step in steps:
    print(step)

Think: What information do I need?
Act: Call get_price('plum cake')
Observe: Function returns Rs 500
Think: Calculate the total for 3 cakes
Answer: 3 plum cakes cost Rs 1500


In [7]:
examples = {
    "GST calculator": "Traditional AI / software",
    "Instagram caption generator": "Generative AI",
    "Order checking and saving assistant": "Agentic AI"
}

for task, level in examples.items():
    print(task, "->", level)

GST calculator -> Traditional AI / software
Instagram caption generator -> Generative AI
Order checking and saving assistant -> Agentic AI


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the assistant for {business}. Answer in under 30 words."),
    ("human", "{question}")
])

In [10]:
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "business": "Sweet Crumbs Home Bakery",
    "question": "Do you make eggless cakes?"
})

print(answer)

Yes, we offer a variety of egg‑free cakes—from chocolate to vanilla—all made with fresh, natural ingredients. Let us know your flavor preference!


In [16]:
SYSTEM = """ROLE:
You are the friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Plum cake Rs 500.
Chocolate cake Rs 650.
Cupcake Rs 80 each.
Orders require 24 hours advance notice.
All cakes have eggless versions.

STRICT RULES:
1. Always use the word eggless when discussing eggless cakes.
2. Never promise same-day orders.
3. Clearly mention 24 hours advance notice.
4. Never confirm delivery to any location.
5. For delivery questions, always reply:
   "I'll check with the owner."
6. Never invent prices or business information.
7. Calculate totals carefully.

FORMAT:
Under 50 words.
Friendly English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

In [12]:
history = []

def ask(question: str) -> str:

    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

In [13]:
print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))

print(ask("What's my name?"))

Hi Ravi! Yes, all our cakes—including plum, chocolate, and cupcakes—have eggless versions. Just give us a 24‑hour notice to place your order. Let me know which you'd like, and I'll calculate the total for you.
Your name is Ravi.


In [17]:
%%writefile tests.csv
question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24
Do you deliver to Kakinada?,owner
What is the price of 6 cupcakes?,480

Overwriting tests.csv


In [18]:
import csv

def run_tests(path="tests.csv"):

    rows = list(csv.DictReader(open(path)))

    passed = 0

    for row in rows:

        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()

        ok = expected in answer.lower()

        if ok:
            passed += 1
            print("PASS |", row["question"])
        else:
            print("FAIL |", row["question"], "->", answer)

    total = len(rows)

    print(f"\n{passed}/{total} passed ({passed/total:.0%})")

run_tests()

PASS | How much is a plum cake?
PASS | Do you make eggless cakes?
PASS | Can I get a cake today evening?
PASS | Do you deliver to Kakinada?
PASS | What is the price of 6 cupcakes?

5/5 passed (100%)


In [19]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, we offer eggless versions of all our cakes. Please place your order at least 24 hours in advance.


In [20]:
customer_questions = [
    "Do you have eggless chocolate cake?",
    "Can I order a cake for this evening?",
    "Do you deliver to Kakinada?",
    "What is the price of 6 cupcakes?"
]

for question in customer_questions:
    print("\nCustomer:", question)
    print("Assistant:", ask(question))


Customer: Do you have eggless chocolate cake?
Assistant: Yes, we offer an eggless chocolate cake for Rs 650. Please place your order at least 24 hours in advance. Let me know how many you’d like!

Customer: Can I order a cake for this evening?
Assistant: I’m sorry, but we can’t take same‑day orders. All cakes need a 24‑hour advance notice. If you place your order now, we can prepare it for tomorrow. Let me know what you’d like!

Customer: Do you deliver to Kakinada?
Assistant: I'll check with the owner.

Customer: What is the price of 6 cupcakes?
Assistant: Six cupcakes cost Rs 480 (₹80 each). Let me know if you’d like to order!


In [21]:
EV_SYSTEM = """You are a helpful EV Customer and Service Assistant.

Your responsibilities:
1. Answer general EV customer questions.
2. Explain basic vehicle maintenance.
3. Guide customers about service requirements.
4. Help customers understand battery and charging issues.

Rules:
Never invent service prices or appointment availability.
If information is missing, ask the customer to contact the service team.
Give simple and clear answers."""

ev_prompt = ChatPromptTemplate.from_messages([
    ("system", EV_SYSTEM),
    ("human", "{question}")
])

ev_chain = ev_prompt | llm | StrOutputParser()

In [22]:
questions = [
    "How can I improve my EV battery life?",
    "What should I do if my EV is not charging?",
    "How often should I service my EV?",
    "Can you tell me the service appointment price?"
]

for question in questions:

    print("\nCustomer:", question)

    answer = ev_chain.invoke({
        "question": question
    })

    print("EV Assistant:", answer)


Customer: How can I improve my EV battery life?
EV Assistant: Here are a few simple ways to help your EV battery stay healthy for longer:

1. **Keep the charge level between 20 % and 80 %**  
   - Full 100 % charges are fine for occasional long trips, but daily use is best at 80 % or less.  
   - Many vehicles let you set a “daily charge limit” in the app or on‑board menu.

2. **Avoid extreme temperatures**  
   - Heat and cold both shorten battery life.  
   - Park in a shaded spot or a garage when possible.  
   - If you’re in a hot climate, use the pre‑conditioning feature to cool the battery before you drive.

3. **Use regenerative braking**  
   - This recaptures energy and reduces the need for high‑power charging.  
   - Most EVs automatically use it, but driving in “Eco” or “Regenerative” mode can boost its use.

4. **Don’t over‑charge or under‑charge**  
   - Let the battery sit at 100 % for long periods (e.g., overnight) only if the manufacturer recommends it.  
   - Similarl

In [23]:
my_question = input("Enter your EV customer question: ")

response = ev_chain.invoke({
    "question": my_question
})

print("\nEV Assistant:", response)

Enter your EV customer question: what is my vehicle battery life present?

EV Assistant: I don’t have access to the real‑time data for your specific vehicle, so I can’t tell you the exact battery level right now.  
Here’s what you can do to check it yourself:

1. **On‑board display** – Most EVs show the remaining charge (in kWh or as a percentage) on the central screen or instrument cluster.  
2. **Mobile app** – If your car has a companion app, it usually reports the current state of charge and estimated range.  
3. **Charging station** – When you plug in, the charger’s screen often displays the battery’s current charge level.

If you’re still unsure or want a more detailed health check, please reach out to your local service center or the dealership. They can run a diagnostic and give you the precise battery status.


In [24]:
llm_creative = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

creative_chain = ev_prompt | llm_creative | StrOutputParser()

question = "Give me three creative ways to improve EV customer experience."

print(creative_chain.invoke({
    "question": question
}))

Here are three fresh ideas that can make the EV experience feel smoother, smarter, and more enjoyable for your customers:

1. **“Drive‑Day Concierge” Mobile Experience**  
   - **What it is**: A mobile app feature that turns a simple trip into a personalized journey.  
   - **How it works**:  
     - The app suggests the best charging stations along the route, factoring in the driver’s real‑time battery level, preferred charging speed, and even nearby attractions.  
     - It can pre‑book a charging slot and send a notification when the spot is ready, so the driver never has to wait.  
     - While waiting, the app offers curated local deals (e.g., coffee, fast‑food, or tourist spots) and can even play a short playlist or podcast to keep the driver entertained.  
   - **Why it matters**: Reduces anxiety about range and charging, and turns downtime into a pleasant, low‑effort part of the trip.

2. **Community‑Powered “EV Swap‑Out” Events**  
   - **What it is**: Organize local meet‑ups 

In [25]:
%%writefile ev_tests.csv
question,must_include
How can I improve EV battery life?,battery
What should I do if my EV is not charging?,charging
How often should I service my EV?,service
Can you tell me the service appointment price?,confirm
Why is my EV range decreasing?,range
How can I charge my EV safely?,charge
What should I do if my EV overheats?,service
Can I book a service appointment?,service
How do I maintain my EV tyres?,tyres
Where is the nearest service centre?,service

Writing ev_tests.csv


In [26]:
import csv

def run_ev_tests(path="ev_tests.csv", model_chain=None):

    if model_chain is None:
        model_chain = ev_chain

    with open(path, newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:

        answer = model_chain.invoke({
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        ok = expected in answer.lower()

        if ok:
            passed += 1
            print("PASS |", row["question"])
        else:
            print("FAIL |", row["question"], "->", answer)

    total = len(rows)
    print(f"\nPass rate: {passed}/{total} ({passed/total:.0%})")

    return passed / total if total else 0

run_ev_tests()

PASS | How can I improve EV battery life?
PASS | What should I do if my EV is not charging?
PASS | How often should I service my EV?
FAIL | Can you tell me the service appointment price? -> I’m sorry, but I don’t have the pricing details for service appointments. Please reach out to our service team or visit the service section on our website for the most accurate and up‑to‑date information.
PASS | Why is my EV range decreasing?
PASS | How can I charge my EV safely?
PASS | What should I do if my EV overheats?
PASS | Can I book a service appointment?
PASS | How do I maintain my EV tyres?
PASS | Where is the nearest service centre?

Pass rate: 9/10 (90%)


0.9

In [27]:
EV_SYSTEM += """

ADDITIONAL RULES:
1. Never invent battery specifications.
2. Never invent service centre locations.
3. Never promise appointment availability.
4. Give safety precautions for charging problems.
5. If exact information is unavailable, ask the service team to confirm.
"""

In [28]:
ev_prompt = ChatPromptTemplate.from_messages([
    ("system", EV_SYSTEM),
    ("human", "{question}")
])

ev_chain = ev_prompt | llm | StrOutputParser()

improved_rate = run_ev_tests()

PASS | How can I improve EV battery life?
PASS | What should I do if my EV is not charging?
PASS | How often should I service my EV?
FAIL | Can you tell me the service appointment price? -> I’m sorry, but I don’t have the pricing details for service appointments. Please reach out to our service team or visit the official website for the most accurate and up‑to‑date information.
PASS | Why is my EV range decreasing?
PASS | How can I charge my EV safely?
PASS | What should I do if my EV overheats?
PASS | Can I book a service appointment?
PASS | How do I maintain my EV tyres?
PASS | Where is the nearest service centre?

Pass rate: 9/10 (90%)


In [29]:
llm_creative = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

creative_chain = ev_prompt | llm_creative | StrOutputParser()

print("RUN 1")
rate1 = run_ev_tests(model_chain=creative_chain)

print("\nRUN 2")
rate2 = run_ev_tests(model_chain=creative_chain)

print("\nRun 1:", rate1)
print("Run 2:", rate2)

RUN 1
PASS | How can I improve EV battery life?
PASS | What should I do if my EV is not charging?
PASS | How often should I service my EV?
FAIL | Can you tell me the service appointment price? -> I’m sorry, but I don’t have the current pricing for service appointments on hand. Please reach out to your local service center or visit our website’s service page for the most up-to-date quote and booking options. If you need help finding a contact or have any other questions, let me know!
PASS | Why is my EV range decreasing?
PASS | How can I charge my EV safely?
PASS | What should I do if my EV overheats?
PASS | Can I book a service appointment?
PASS | How do I maintain my EV tyres?
PASS | Where is the nearest service centre?

Pass rate: 9/10 (90%)

RUN 2
PASS | How can I improve EV battery life?
PASS | What should I do if my EV is not charging?
PASS | How often should I service my EV?
FAIL | Can you tell me the service appointment price? -> I don’t have the exact pricing details for servic

In [30]:
ev_jobs = {
    "Calculate service bill": "Plain Python code",
    "Explain EV battery maintenance": "One LLM call",
    "Understand customer issue and choose appropriate service tools": "Agent"
}

for job, approach in ev_jobs.items():
    print(job, "->", approach)

Calculate service bill -> Plain Python code
Explain EV battery maintenance -> One LLM call
Understand customer issue and choose appropriate service tools -> Agent


In [31]:
def get_service_price(service_type: str) -> str:
    """
    Return the available price information for an EV service.
    """

    prices = {
        "general_service": 1500,
        "battery_check": 800,
        "tyre_check": 300
    }

    service_type = service_type.lower()

    if service_type in prices:
        return f"Estimated service price: Rs {prices[service_type]}"

    return "Price not available. Please contact the service team."


def check_service_availability(service_type: str) -> str:
    """
    Check whether a service type is listed for appointment enquiry.
    """

    available_services = [
        "general_service",
        "battery_check",
        "tyre_check"
    ]

    if service_type.lower() in available_services:
        return "Service enquiry accepted. Appointment availability must be confirmed by the service team."

    return "Service type not found. Please contact the service team."


print(get_service_price("battery_check"))
print(check_service_availability("general_service"))

Estimated service price: Rs 800
Service enquiry accepted. Appointment availability must be confirmed by the service team.
